In [5]:
"""
Selecting the best code per (n, k, q) from the distance results
================================================================

What this script does
---------------------
It reads the output of the GAP distance script for one block length n,

    data/data_{n}n_{k}k_{q}q_{w}w/dis_n{n}_{k}k_{q}q_{w}wait.csv

(columns f, g, f_no_neg, g_no_neg, alpha, beta, gamma, n, k_gamma0,
k_gamma, d0, d, ratio0, ratio; d0 and ratio0 refer to the untwisted torus
gamma = 0 with the same f, g, alpha, beta, and d and ratio to the twisted
torus), and ranks the codes by the rule used for the tables of the paper:

    1. largest figure of merit  k d^2 / n  (the larger of ratio0 and ratio);
    2. among codes with equal k d^2 / n, the most local stabilizers, i.e.
       the smallest stabilizer range

           loc = (max x-exponent - min x-exponent)
               + (max y-exponent - min y-exponent)

       taken over all monomials of f and g (Definition 1 of the paper).

The first row of the printed table is the entry that goes into the paper.
Note: when best_ratio comes from ratio0, the best code is the untwisted
torus of that row (gamma = 0, distance d0), and the table must be written
accordingly.

Reference
---------
This script accompanies

    M. Halla, "Qudit Twisted-Torus Codes in the Bivariate Bicycle Framework",
    arXiv:2602.04443, https://arxiv.org/abs/2602.04443

Please cite this paper if you use the script or the codes found with it.

Software: Python 3.11 with pandas.
Copyright (c) 2026 Mourad Halla.  Licence: MIT (see LICENSE).
"""

import pandas as pd
import re

##############################################
# Parameters
##############################################
n = 42         # block length (number of physical qudits)

q = 3           # local dimension of the qudits

w = 6           # check weight; only used in the file names

k_targ = 4      # number of logical qudits; only used in the file names

r = 1           # rows with k d^2/n below this value are ignored

best5 = 5      # number of best codes shown

data_dir = f"data/data_{n}n_{k_targ}k_{q}q_{w}w"
df = pd.read_csv(f"{data_dir}/dis_n{n}_{k_targ}k_{q}q_{w}wait.csv")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# Make numeric copies of ratio0, ratio (the GAP script writes "fail" when
# no distance could be computed; these become NaN and are ignored)
df_num = df.copy()
df_num["ratio0"] = pd.to_numeric(df_num["ratio0"], errors="coerce")
df_num["ratio"]  = pd.to_numeric(df_num["ratio"],  errors="coerce")

##############################################
# Filter: keep rows with ratio0 ≥ r or ratio ≥ r
##############################################
mask = (
    (df_num["ratio0"] >= r) |
    (df_num["ratio"]  >= r)
)

df_sel = df.loc[mask].reset_index().rename(columns={"index": "orig_index"})
df_sel_num = df_num.loc[mask].reset_index(drop=True)

##############################################
# Parsing polynomials (exponents in x,y)
#
# The polynomials are stored as strings such as "x^2*y^-3 + 1 + x^-2".
# The three helpers below read the exponent of x and of y in every term;
# coefficients are not needed for the stabilizer range and are dropped.
##############################################

def clean_poly_string(s):
    """Keep only x,y, digits, +,-,^,* and spaces."""
    if s is None:
        return ""
    s = str(s)
    allowed = set("xyXY0123456789+-^* ")
    return "".join(ch for ch in s if ch in allowed)

def split_terms(expr):
    """
    Split a polynomial string into terms, respecting exponent signs.
    Example: "x + x*y^2 + 1" -> ["x", "+x*y^2", "+1"]
    A sign directly after '^' belongs to an exponent and does not start
    a new term.
    """
    expr = expr.replace(' ', '')
    if not expr:
        return []
    terms = []
    current = ''
    for i, ch in enumerate(expr):
        # new term at + or -, unless it's the sign of exponent (after ^)
        if ch in '+-' and i > 0 and expr[i-1] != '^':
            terms.append(current)
            current = ch
        else:
            current += ch
    if current:
        terms.append(current)
    return terms

def parse_term(term):
    """
    term like: -3*x^2*y, x*y^2, y^3, 5, x, -y^3, ...
    returns (ax, ay) so that term ∼ x^ax * y^ay
    """
    t = term.replace(' ', '')
    if not t:
        return None

    # drop leading sign, we don't care about coefficient sign for locality
    if t[0] in '+-':
        t = t[1:]

    # split by '*'
    factors = t.split('*') if t else []

    # optional integer coefficient at front
    if factors and re.fullmatch(r'-?\d+', factors[0]):
        factors = factors[1:]

    ax = 0
    ay = 0
    for f in factors:
        if not f:
            continue
        if f[0] in ('x', 'y'):
            var = f[0]
            exp = 1
            if len(f) > 1:
                if f[1] != '^':
                    # malformed, ignore for locality
                    continue
                exp_str = f[2:]
                if exp_str == '':
                    continue
                exp = int(exp_str)
            if var == 'x':
                ax += exp
            else:
                ay += exp
        else:
            continue
    return ax, ay

def poly_exponents(poly_str):
    """Return list of (ax, ay) exponent pairs from a polynomial string."""
    s = clean_poly_string(poly_str)
    terms = split_terms(s)
    exps = []
    for t in terms:
        parsed = parse_term(t)
        if parsed is not None:
            exps.append(parsed)
    # if polynomial is constant, treat it as exponent (0,0)
    if not exps:
        exps = [(0, 0)]
    return exps

##############################################
# Locality = stabilizer range:
#   (max x-exp - min x-exp) + (max y-exp - min y-exp)
# over all monomials of f and g, on the original f, g
# (the same quantity as in Ref. [8]: the extent of a check on the lattice;
#  smaller means more local)
##############################################

def locality_row(row):
    exps_f = poly_exponents(row["f"])
    exps_g = poly_exponents(row["g"])

    xs = []
    ys = []

    for (a, b) in (exps_f + exps_g):
        xs.append(a)
        ys.append(b)

    dx = max(xs) - min(xs)
    dy = max(ys) - min(ys)
    return dx + dy   # smaller = more local

##############################################
# Compute locality and best_ratio
##############################################

df_sel["loc"] = df_sel.apply(locality_row, axis=1)

# best_ratio = max(ratio0, ratio) row-wise
df_sel["ratio0_num"] = df_sel_num["ratio0"]
df_sel["ratio_num"]  = df_sel_num["ratio"]
df_sel["best_ratio"] = df_sel[["ratio0_num", "ratio_num"]].max(axis=1)

##############################################
# Sort and keep the best codes
#   1) highest best_ratio (kd^2/n)
#   2) if tie, smallest locality (most local stabilizers)
##############################################

df_sorted = df_sel.sort_values(
    ["best_ratio", "loc"],
    ascending=[False, True]
)

df_best = df_sorted.head(best5)

##############################################
# Display (without f_no_neg, g_no_neg)
##############################################

def fmt2(x):
    """Print a number with two decimals; leave non-numbers as they are."""
    try:
        val = float(x)
    except (TypeError, ValueError):
        return x
    return f"{val:.2f}"

cols_to_show = [
    "f", "g",
    "alpha", "beta", "gamma",
    "n", "k_gamma0", "k_gamma",
    "d0", "d", "ratio0", "ratio",
    "loc",
]

df_best_display = df_best.copy()
df_best_display["ratio0"] = df_best["ratio0_num"]
df_best_display["ratio"]  = df_best["ratio_num"]

df_best_display[cols_to_show].style.hide(axis="index") \
    .format({"ratio0": fmt2, "ratio": fmt2}) \
    .set_table_styles([
        {"selector": "th, td", "props": [("font-size", "10pt")]}
    ])

f,g,alpha,beta,gamma,n,k_gamma0,k_gamma,d0,d,ratio0,ratio,loc
1 + y^-2 + x^-1*y^-2,x^2*y + 1 + x^-1*y^-3,3,7,-2,42,2,4,7,8,2.33,6.10,7
1 + y^-2 + x^-1*y^-2,x^2*y + 1 + x^-1*y^-3,3,7,1,42,2,4,7,8,2.33,6.10,7
1 + x^-1*y + x^-2*y^3,x^2*y^-3 + 1 + x^-2*y^-1,7,3,-3,42,4,4,6,8,3.43,6.10,10
1 + x^-1*y + x^-2*y^3,x^2*y^-3 + 1 + x^-2*y^-1,7,3,-2,42,4,4,6,8,3.43,6.10,10
1 + x^-1*y + x^-2*y^3,x^2*y^-3 + 1 + x^-2*y^-1,7,3,4,42,4,4,6,8,3.43,6.10,10
